In [ ]:
#!/usr/bin/env python3
"""
RSA Lag Analysis — Per-Region, with Inline Semantic Similarity
---------------------------------------------------------------
Runs the RSA lag pipeline SEPARATELY for each of 6 brain regions:
    LHP, RHP, LEC, REC, LPC, RPC

For every trial, for every pair of clean recalls (i, j) with i < j:
  - Build retrieval vector from channels belonging to THAT region only
  - RSA_r       : Pearson r between vector_i and vector_j
  - output_lag  : j - i  (signed output-position lag)
  - T_lag       : |serial_pos_i - serial_pos_j|
  - SP_lag      : Euclidean distance between store locations
  - semantic_sim: Word2Vec cosine similarity between word_i and word_j

Output columns:
  subject, session, experiment, trial, region,
  output_pos_i, output_pos_j, output_lag,
  word_i, word_j,
  serial_pos_i, serial_pos_j, T_lag, SP_lag,
  RSA_r, n_channels, semantic_sim

One master CSV per experiment per region:
  ./rsa_lag_results/ALL_SUBJECTS_{exp}_{region}_rsa_lag.csv

One combined CSV per experiment (all regions stacked):
  ./rsa_lag_results/ALL_SUBJECTS_{exp}_allregions_rsa_lag.csv
"""

import warnings
import traceback
from pathlib import Path
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from scipy.spatial.distance import euclidean

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

EXPERIMENTS = ['DBOY1', 'EFRCourierReadOnly', 'EFRCourierOpenLoop']

INPUT_DIRS = {
    'DBOY1':               Path('./subject_results_DBOY1'),
    'EFRCourierReadOnly':  Path('./subject_results_EFRCourierReadOnly'),
    'EFRCourierOpenLoop':  Path('./subject_results_EFRCourierOpenLoop'),
}

OUTPUT_DIR = Path('./rsa_lag_results')
OUTPUT_DIR.mkdir(exist_ok=True)

# All six regions — each run independently
REGIONS = ['LHP', 'RHP', 'LEC', 'REC', 'LPC', 'RPC']

# Region → hemisphere tag (for downstream analysis)
HEMISPHERE = {r: ('left' if r.startswith('L') else 'right') for r in REGIONS}

# Region → region-group tag (HP / EC / PC)
REGION_GROUP = {
    'LHP': 'HP', 'RHP': 'HP',
    'LEC': 'EC', 'REC': 'EC',
    'LPC': 'PC', 'RPC': 'PC',
}

N_FREQS       = 18
RET_OSC_COLS  = [f'ret_osc_f{i:02d}' for i in range(N_FREQS)]

# ---- Word2Vec ---------------------------------------------------------------
# Set to None to skip semantic similarity (useful for quick testing)
WORD2VEC_PATH = Path('/home1/noaherz/word2vec/GoogleNews-vectors-negative300.bin.gz')

OUTPUT_COLS = [
    'subject', 'session', 'experiment', 'trial', 'region', 'hemisphere',
    'region_group',
    'output_pos_i', 'output_pos_j', 'output_lag',
    'word_i', 'word_j',
    'serial_pos_i', 'serial_pos_j', 'T_lag', 'SP_lag',
    'RSA_r', 'n_channels', 'semantic_sim',
]

# ============================================================================
# WORD2VEC LOADER + SIMILARITY
# ============================================================================

def load_word2vec(path: Path):
    """Load KeyedVectors. Returns None if path missing or import fails."""
    if path is None or not path.exists():
        print(f"  ⚠  Word2Vec model not found at {path}. "
              f"semantic_sim will be NaN.")
        return None
    try:
        import gensim.models as gensim_models
        print(f"  Loading Word2Vec from {path} …")
        model = gensim_models.KeyedVectors.load_word2vec_format(
            str(path), binary=True)
        try:
            vs = len(model)
        except TypeError:
            vs = len(model.vocab)
        print(f"  ✓ Word2Vec loaded — vocab: {vs:,}")
        return model
    except Exception as e:
        print(f"  ✗ Word2Vec load failed: {e}. semantic_sim will be NaN.")
        return None


def case_insensitive_similarity(word1: str, word2: str, model) -> Optional[float]:
    """
    Cosine similarity between two words; tries all lower/upper combinations
    and returns the maximum.  Returns None if neither is in vocabulary.
    (Follows Semantic_sim_word2vec.ipynb convention.)
    """
    cases = [
        (word1.lower(), word2.lower()),
        (word1.lower(), word2.upper()),
        (word1.upper(), word2.lower()),
        (word1.upper(), word2.upper()),
    ]
    sims = []
    for w1, w2 in cases:
        try:
            sims.append(model.similarity(w1, w2))
        except KeyError:
            continue
    return float(max(sims)) if sims else None


def build_similarity_cache(words: set, model) -> dict:
    """
    Pre-compute Word2Vec similarities for all unique unordered word pairs.
    Returns dict keyed by frozenset({w1, w2}).
    """
    if model is None:
        return {}
    unique_words = sorted(w for w in words if isinstance(w, str))
    n = len(unique_words)
    print(f"    Building semsim cache: {n} words → {n*(n-1)//2} pairs …")
    cache = {}
    for i, w1 in enumerate(unique_words):
        for w2 in unique_words[i:]:
            key = frozenset({w1, w2})
            if key not in cache:
                sim = case_insensitive_similarity(w1, w2, model)
                cache[key] = sim if sim is not None else np.nan
    return cache


# ============================================================================
# VECTOR + STATISTICS HELPERS
# ============================================================================

def build_retrieval_vector(recall_rows: pd.DataFrame,
                           channel_index) -> np.ndarray:
    """
    Build a retrieval IRASA vector aligned to channel_index.
    Missing channels are filled with NaN (handled pairwise in safe_pearsonr).
    """
    ch_df = (
        recall_rows
        .drop_duplicates(subset='channel_idx')
        .set_index('channel_idx')
        .reindex(channel_index)
    )
    mat = ch_df[RET_OSC_COLS].values   # (N_ch, 18)
    return mat.flatten()


def safe_pearsonr(v1: np.ndarray, v2: np.ndarray) -> float:
    if len(v1) != len(v2):
        return np.nan
    mask = np.isfinite(v1) & np.isfinite(v2)
    if mask.sum() < 3:
        return np.nan
    r, _ = pearsonr(v1[mask], v2[mask])
    return float(r)


def safe_euclidean(x1, z1, x2, z2) -> float:
    if any(not np.isfinite(v) for v in (x1, z1, x2, z2)):
        return np.nan
    return float(euclidean([x1, z1], [x2, z2]))


def extract_scalar(series: pd.Series, field: str, context: str):
    unique_vals = series.dropna().unique()
    if len(unique_vals) > 1:
        warnings.warn(
            f"[{context}] '{field}' has {len(unique_vals)} distinct values "
            f"({unique_vals[:3]}…). Taking first.")
    return series.iloc[0]


# ============================================================================
# TRIAL-LEVEL PROCESSING  (for one region)
# ============================================================================

def process_trial_region(trial_df: pd.DataFrame,
                         region: str,
                         sim_cache: dict) -> List[Dict]:
    """
    trial_df  : rows for one (subject, session, trial) filtered to `region`
    region    : e.g. 'LHP'
    sim_cache : frozenset → float (may be empty dict if no Word2Vec)
    """
    rows = []

    output_positions = sorted(
        trial_df['recall_output_position'].unique(),
        key=lambda x: int(x),
    )
    if len(output_positions) < 2:
        return rows

    channel_index = sorted(trial_df['channel_idx'].unique(), key=int)

    pos_data: Dict[int, Dict] = {}
    sample_row = trial_df.iloc[0]

    for op in output_positions:
        op_rows = trial_df[trial_df['recall_output_position'] == op]
        ctx = (f"subj={sample_row['subject']} sess={sample_row['session']} "
               f"trial={sample_row['trial']} region={region} op={op}")

        word       = extract_scalar(op_rows['recalled_word'],   'recalled_word',   ctx)
        serial_pos = extract_scalar(op_rows['serial_position'], 'serial_position', ctx)
        store_x    = extract_scalar(op_rows['store_x'],         'store_x',         ctx)
        store_z    = extract_scalar(op_rows['store_z'],         'store_z',         ctx)

        vec  = build_retrieval_vector(op_rows, channel_index)
        n_ch = op_rows['channel_idx'].nunique()

        pos_data[op] = {
            'vector':     vec,
            'word':       word,
            'serial_pos': serial_pos,
            'store_x':    store_x,
            'store_z':    store_z,
            'n_channels': n_ch,
        }

    subject    = sample_row['subject']
    session    = sample_row['session']
    experiment = sample_row['experiment']
    trial      = sample_row['trial']

    for idx_i, op_i in enumerate(output_positions):
        for op_j in output_positions[idx_i + 1:]:
            d_i = pos_data[op_i]
            d_j = pos_data[op_j]

            output_lag = int(op_j) - int(op_i)
            T_lag      = abs(int(d_i['serial_pos']) - int(d_j['serial_pos']))
            SP_lag     = safe_euclidean(d_i['store_x'], d_i['store_z'],
                                        d_j['store_x'], d_j['store_z'])
            rsa_r      = safe_pearsonr(d_i['vector'], d_j['vector'])

            n_channels = min(d_i['n_channels'], d_j['n_channels'])
            if d_i['n_channels'] != d_j['n_channels']:
                warnings.warn(
                    f"Channel count differs: op{op_i}={d_i['n_channels']} "
                    f"vs op{op_j}={d_j['n_channels']}. Reporting min.")

            # --- semantic similarity ---
            w_i = str(d_i['word']).lower() if pd.notna(d_i['word']) else None
            w_j = str(d_j['word']).lower() if pd.notna(d_j['word']) else None
            if w_i and w_j and sim_cache:
                sem_sim = sim_cache.get(frozenset({w_i, w_j}), np.nan)
            else:
                sem_sim = np.nan

            rows.append({
                'subject':       subject,
                'session':       session,
                'experiment':    experiment,
                'trial':         trial,
                'region':        region,
                'hemisphere':    HEMISPHERE[region],
                'region_group':  REGION_GROUP[region],
                'output_pos_i':  op_i,
                'output_pos_j':  op_j,
                'output_lag':    output_lag,
                'word_i':        d_i['word'],
                'word_j':        d_j['word'],
                'serial_pos_i':  d_i['serial_pos'],
                'serial_pos_j':  d_j['serial_pos'],
                'T_lag':         T_lag,
                'SP_lag':        SP_lag,
                'RSA_r':         rsa_r,
                'n_channels':    n_channels,
                'semantic_sim':  sem_sim,
            })

    return rows


# ============================================================================
# PER-EXPERIMENT RUNNER
# ============================================================================

def run_experiment(exp: str, w2v_model):
    print(f"\n{'='*70}")
    print(f"EXPERIMENT: {exp}")
    print(f"{'='*70}")

    input_dir  = INPUT_DIRS.get(exp)
    if input_dir is None:
        print(f"  ✗ No INPUT_DIR configured for '{exp}'.")
        return

    input_path = input_dir / f"ALL_SUBJECTS_{exp}_irasa_channel_wide.csv"
    if not input_path.exists():
        print(f"  ✗ Master CSV not found: {input_path}")
        return

    print(f"  Loading {input_path} …")
    df = pd.read_csv(input_path)
    print(f"  Loaded {len(df):,} rows | "
          f"{df['subject'].nunique()} subjects | "
          f"{df['session'].nunique()} sessions")

    # Cast channel_idx to int for consistent sorting
    df['channel_idx'] = df['channel_idx'].astype(int)

    # Build Word2Vec cache once per experiment (all words across all regions)
    all_words_lower = set(
        df['recalled_word'].dropna().astype(str).str.lower().unique()
    )
    sim_cache = build_similarity_cache(all_words_lower, w2v_model)

    all_region_dfs = []   # collect all regions for the combined file

    for region in REGIONS:
        print(f"\n  {'─'*60}")
        print(f"  Region: {region}")

        region_df = df[df['region'] == region].copy()
        if region_df.empty:
            print(f"  ✗ No rows for region {region} — skipping")
            continue

        print(f"  Rows: {len(region_df):,}")

        all_rows = []
        groups   = region_df.groupby(['subject', 'session', 'trial'])
        n_groups = len(groups)

        for g_idx, ((subj, sess, trial), trial_df) in enumerate(groups):
            if g_idx % 200 == 0:
                print(f"    Processing trial group {g_idx}/{n_groups} …")
            try:
                rows = process_trial_region(trial_df, region, sim_cache)
                all_rows.extend(rows)
            except Exception as e:
                print(f"    FAILED [{subj} sess={sess} trial={trial}]: {e}")
                traceback.print_exc()
                continue

        if not all_rows:
            print(f"  ✗ No pairs generated for region {region}")
            continue

        result_df = pd.DataFrame(all_rows, columns=OUTPUT_COLS)
        all_region_dfs.append(result_df)

        # Per-region output
        out_path = OUTPUT_DIR / f"ALL_SUBJECTS_{exp}_{region}_rsa_lag.csv"
        result_df.to_csv(out_path, index=False)

        n_nan_sem = result_df['semantic_sim'].isna().sum()
        print(f"  ✓ {region}: {len(result_df):,} pairs | "
              f"RSA NaN={result_df['RSA_r'].isna().mean()*100:.1f}% | "
              f"SemSim NaN={n_nan_sem/len(result_df)*100:.1f}% | "
              f"→ {out_path.name}")

    # Combined output (all regions stacked)
    if all_region_dfs:
        combined = pd.concat(all_region_dfs, ignore_index=True)
        comb_path = OUTPUT_DIR / f"ALL_SUBJECTS_{exp}_allregions_rsa_lag.csv"
        combined.to_csv(comb_path, index=False)
        print(f"\n  ✓ Combined all-regions CSV → {comb_path}")
        print(f"    Total pairs: {len(combined):,} | "
              f"Regions: {combined['region'].unique().tolist()}")


# ============================================================================
# MAIN
# ============================================================================

if __name__ == '__main__':
    w2v_model = load_word2vec(WORD2VEC_PATH)

    for exp in EXPERIMENTS:
        run_experiment(exp, w2v_model)

    print(f"\n{'='*70}")
    print("✓ ALL EXPERIMENTS COMPLETE")
    print(f"{'='*70}")

  Loading Word2Vec from /home1/noaherz/word2vec/GoogleNews-vectors-negative300.bin.gz …


In [1]:
#!/usr/bin/env python3
"""
LMM Analysis: RSA_r → T_lag / SP_lag
Across all region groups (HP, EC, PC) × hemispheres (left, right)
----------------------------------------------------------------------
Input : ./rsa_lag_results/ALL_SUBJECTS_{exp}_allregions_rsa_lag.csv
        (output of rsa_lag_by_region.py)

For each OUTCOME in [T_lag, SP_lag]:
  For each REGION_GROUP in [HP, EC, PC]:

    Model 1 (bare):
        outcome ~ RSA_r + experiment_dummy   + (1|subj/sess)

    Model 2 (controlled):
        outcome ~ RSA_r + output_lag + semantic_sim + experiment_dummy
                                                     + (1|subj/sess)

    Model 3 (hemisphere main effect):
        outcome ~ RSA_r + hemisphere_dummy + experiment_dummy
                                                     + (1|subj/sess)

    Model 4 (hemisphere interaction):
        outcome ~ RSA_r * hemisphere_dummy + experiment_dummy
                                                     + (1|subj/sess)

    Model 5 (full + hemisphere interaction):
        outcome ~ RSA_r * hemisphere_dummy + output_lag
                  + semantic_sim + experiment_dummy  + (1|subj/sess)

hemisphere_dummy: left=0 (reference), right=1
experiment_dummy: DBOY1=0 (reference), others=1 (if multiple experiments)

Outputs:
  ./rsa_lag_results/LMM_{outcome}_{region_group}_results.csv
  ./rsa_lag_results/LMM_{outcome}_{region_group}_results.txt
  ./rsa_lag_results/plots/forest_{outcome}_{region_group}.png
  ./rsa_lag_results/plots/interaction_{outcome}_{region_group}.png
  ./rsa_lag_results/plots/summary_heatmap.png
"""

import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from scipy.stats import pearsonr
from statsmodels.regression.mixed_linear_model import MixedLM
from statsmodels.stats.multitest import fdrcorrection

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

EXPERIMENTS   = ['DBOY1', 'EFRCourierReadOnly', 'EFRCourierOpenLoop']
INPUT_DIR     = Path('./rsa_lag_results')
OUTPUT_DIR    = Path('./rsa_lag_results')
PLOT_DIR      = OUTPUT_DIR / 'plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

OUTCOMES      = ['T_lag', 'SP_lag']
REGION_GROUPS = ['HP', 'EC', 'PC']

# ---- Palette (deep, neuroscience-inspired) ----------------------------------
# Region groups
RG_COLORS = {
    'HP': '#1A3A6B',   # deep navy
    'EC': '#5C3317',   # deep brown
    'PC': '#2D6B3A',   # deep forest green
}
# Hemispheres
HEMI_COLORS = {
    'left':  '#2166AC',   # steel blue
    'right': '#B2182B',   # crimson
}
# Outcomes
OUT_COLORS = {
    'T_lag':  '#4A235A',   # deep purple
    'SP_lag': '#1A5276',   # deep teal
}

OUTCOME_LABELS = {
    'T_lag':  'Temporal Lag',
    'SP_lag': 'Spatial Lag',
}
MODEL_LABELS = {
    'Model1': 'Bare',
    'Model2': 'Controlled',
    'Model3': 'Hemisphere\nmain effect',
    'Model4': 'Hemisphere\ninteraction',
    'Model5': 'Full +\ninteraction',
}

# ============================================================================
# LMM FITTING
# ============================================================================

def fit_lmm(df: pd.DataFrame,
            outcome: str,
            pred_cols: List[str],
            label: str) -> Tuple[Optional[object], int]:
    df = df.copy()
    df['subj_sess'] = (df['subject'].astype(str) + '_' +
                       df['session'].astype(str))
    keep = [outcome] + pred_cols + ['subject', 'subj_sess']
    df   = df[keep].dropna()

    if len(df) < 20:
        print(f"    [{label}] Too few rows ({len(df)}) — skipping")
        return None, 0

    formula = f"{outcome} ~ {' + '.join(pred_cols)}"
    print(f"    [{label}] {formula}  |  N={len(df):,}")

    model = MixedLM.from_formula(
        formula,
        data       = df,
        groups     = df['subject'],
        vc_formula = {'subj_sess': '0 + C(subj_sess)'},
    )

    result = None
    for method in ['lbfgs', 'nm', 'powell']:
        try:
            result = model.fit(reml=True, method=method)
            if np.isfinite(result.llf):
                print(f"    [{label}] optimizer={method}  "
                      f"converged={getattr(result,'converged',None)}  "
                      f"llf={result.llf:.4f}  AIC={result.aic:.4f}")
                break
            else:
                print(f"    [{label}] llf=NaN with {method}, trying next …")
        except Exception as e:
            print(f"    [{label}] {method} failed: {e}")
            result = None

    if result is None or not np.isfinite(result.llf):
        print(f"    [{label}] WARNING: fit unsuccessful.")
    return result, len(df)


# ============================================================================
# RESULT EXTRACTION
# ============================================================================

def extract_rows(result,
                 pred_display: Dict[str, str],
                 model_label: str,
                 outcome: str,
                 region_group: str,
                 hemisphere: str = 'combined') -> pd.DataFrame:
    if result is None:
        return pd.DataFrame()
    rows = []
    for col, name in pred_display.items():
        # Fuzzy match for interaction terms (statsmodels renames them)
        matched = None
        if col in result.params.index:
            matched = col
        else:
            hits = [k for k in result.params.index
                    if col.replace(':', ':').lower() in k.lower()]
            if hits:
                matched = hits[0]
        if matched is None:
            print(f"    WARNING: '{col}' not found — skipping")
            continue
        rows.append({
            'outcome':       outcome,
            'region_group':  region_group,
            'hemisphere':    hemisphere,
            'model':         model_label,
            'predictor':     name,
            'col':           col,
            'beta':          result.params[matched],
            'se':            result.bse[matched],
            'z':             result.tvalues[matched],
            'p_raw':         result.pvalues[matched],
            'llf':           result.llf,
            'aic':           result.aic,
            'nobs':          int(result.nobs),
        })
    return pd.DataFrame(rows)


def apply_fdr_within_model(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    _, df['p_fdr'] = fdrcorrection(df['p_raw'].values)
    return df


# ============================================================================
# FORMATTING
# ============================================================================

def sig_stars(p: float) -> str:
    return ('***' if p < 0.001 else
            '**'  if p < 0.01  else
            '*'   if p < 0.05  else
            '†'   if p < 0.10  else '')


def format_block(title: str, rows_df: pd.DataFrame) -> str:
    sep  = '=' * 90
    sep2 = '-' * 90
    hdr  = (f"{'Predictor':<38} {'β':>8} {'SE':>8} {'z':>8} "
            f"{'p_raw':>10} {'p_fdr':>10} {'AIC':>10} {'N':>8} {'sig':>5}")
    lines = [sep, title, sep2, hdr, sep2]
    for _, row in rows_df.iterrows():
        aic_s = f"{row['aic']:>10.2f}" if pd.notna(row.get('aic')) else '       NaN'
        p_fdr = row.get('p_fdr', np.nan)
        lines.append(
            f"{row['predictor']:<38} {row['beta']:>8.4f} {row['se']:>8.4f} "
            f"{row['z']:>8.3f} {row['p_raw']:>10.4f} {p_fdr:>10.4f} "
            f"{aic_s} {int(row['nobs']):>8,} {sig_stars(p_fdr):>5}"
        )
    lines += [sep2,
              'FDR: BH within each model across predictors.',
              '† p<.10  * p<.05  ** p<.01  *** p<.001',
              sep]
    return '\n'.join(lines)


# ============================================================================
# PLOTTING HELPERS
# ============================================================================

def plot_forest(all_results: pd.DataFrame,
                outcome: str,
                region_group: str,
                save_path: Path):
    """
    Forest plot of RSA_r beta ± 95%CI across models and hemispheres.
    """
    df = all_results[
        (all_results['outcome']      == outcome) &
        (all_results['region_group'] == region_group) &
        (all_results['col'].str.contains('RSA_r', na=False))
    ].copy()

    if df.empty:
        print(f"    No RSA_r rows for forest plot {outcome} {region_group}")
        return

    # 95% CI = beta ± 1.96*SE
    df['ci_lo'] = df['beta'] - 1.96 * df['se']
    df['ci_hi'] = df['beta'] + 1.96 * df['se']

    models  = [m for m in ['Model1', 'Model2', 'Model3', 'Model4', 'Model5']
               if m in df['model'].values]
    hemis   = ['left', 'right', 'combined']

    fig, ax = plt.subplots(figsize=(10, max(4, len(models) * 1.4 + 1)))
    fig.patch.set_facecolor('#0D1117')
    ax.set_facecolor('#0D1117')

    y_pos  = 0
    yticks = []
    ylabels = []

    colors_hemi = {
        'left':     HEMI_COLORS['left'],
        'right':    HEMI_COLORS['right'],
        'combined': '#AAAAAA',
    }
    markers_hemi = {'left': 'o', 'right': 's', 'combined': 'D'}

    for m_idx, model in enumerate(models):
        sub = df[df['model'] == model]
        for h in hemis:
            row = sub[sub['hemisphere'] == h]
            if row.empty:
                continue
            row = row.iloc[0]
            col  = colors_hemi[h]
            mkr  = markers_hemi[h]
            ax.errorbar(row['beta'], y_pos,
                        xerr=[[row['beta'] - row['ci_lo']],
                               [row['ci_hi'] - row['beta']]],
                        fmt=mkr, color=col,
                        ecolor=col, elinewidth=1.5,
                        capsize=4, capthick=1.5,
                        markersize=7, zorder=3)
            # significance annotation
            p_fdr = row.get('p_fdr', row['p_raw'])
            stars = sig_stars(p_fdr)
            if stars:
                ax.text(row['ci_hi'] + 0.002, y_pos, stars,
                        color=col, va='center', fontsize=9,
                        fontweight='bold')
            yticks.append(y_pos)
            ylabels.append(f"{MODEL_LABELS.get(model, model)}  [{h}]")
            y_pos -= 1

        # gap between model groups
        y_pos -= 0.4

    ax.axvline(0, color='#666666', lw=1, ls='--', zorder=1)
    ax.set_yticks(yticks)
    ax.set_yticklabels(ylabels, color='#CCCCCC', fontsize=8.5)
    ax.set_xlabel('β (RSA_r)', color='#CCCCCC', fontsize=10)
    ax.tick_params(colors='#CCCCCC')
    for spine in ax.spines.values():
        spine.set_edgecolor('#444444')

    # Legend
    legend_handles = [
        Line2D([0], [0], marker='o', color='w',
               markerfacecolor=HEMI_COLORS['left'],  markersize=8, label='Left'),
        Line2D([0], [0], marker='s', color='w',
               markerfacecolor=HEMI_COLORS['right'], markersize=8, label='Right'),
        Line2D([0], [0], marker='D', color='w',
               markerfacecolor='#AAAAAA',            markersize=8, label='Combined'),
    ]
    ax.legend(handles=legend_handles, loc='upper right',
              facecolor='#1A1A2E', edgecolor='#444444',
              labelcolor='#CCCCCC', fontsize=8)

    ax.set_title(
        f"RSA_r → {OUTCOME_LABELS[outcome]}  |  {region_group}",
        color='#EEEEEE', fontsize=12, fontweight='bold', pad=10)

    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"    ✓ Forest plot → {save_path.name}")


def plot_interaction(all_results: pd.DataFrame,
                     outcome: str,
                     region_group: str,
                     raw_df: pd.DataFrame,
                     save_path: Path):
    """
    Scatter + regression lines showing RSA_r vs outcome for left vs right.
    Uses raw data for the scatter; LMM beta from Model4 for reference line.
    """
    sub_df = raw_df[
        (raw_df['region_group'] == region_group)
    ].copy().dropna(subset=['RSA_r', outcome])

    if sub_df.empty:
        return

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    fig.patch.set_facecolor('#0D1117')
    title_col = '#EEEEEE'

    for ax, hemi in zip(axes, ['left', 'right']):
        ax.set_facecolor('#111827')
        hemi_data = sub_df[sub_df['hemisphere'] == hemi]
        if hemi_data.empty:
            ax.set_visible(False)
            continue

        color = HEMI_COLORS[hemi]

        # Bin RSA_r into deciles for cleaner scatter
        hemi_data = hemi_data.copy()
        hemi_data['rsa_bin'] = pd.qcut(hemi_data['RSA_r'], q=10,
                                        duplicates='drop', labels=False)
        binned = hemi_data.groupby('rsa_bin').agg(
            rsa_mean=('RSA_r', 'mean'),
            out_mean=(outcome, 'mean'),
            out_sem=(outcome, lambda x: x.std() / np.sqrt(len(x))),
        ).reset_index()

        ax.errorbar(binned['rsa_mean'], binned['out_mean'],
                    yerr=binned['out_sem'],
                    fmt='o', color=color, ecolor=color,
                    elinewidth=1.5, capsize=3, markersize=6,
                    alpha=0.85, zorder=3, label='Binned mean ± SEM')

        # OLS trend line on raw data
        x = hemi_data['RSA_r'].values
        y = hemi_data[outcome].values
        m, b = np.polyfit(x, y, 1)
        r, p  = pearsonr(x, y)
        xline = np.linspace(x.min(), x.max(), 200)
        ax.plot(xline, m * xline + b,
                color=color, lw=2, alpha=0.7,
                label=f'r={r:.3f}, p={p:.3f}')

        ax.set_xlabel('RSA_r (neural similarity)', color='#CCCCCC', fontsize=9)
        ax.set_ylabel(OUTCOME_LABELS[outcome], color='#CCCCCC', fontsize=9)
        ax.set_title(f'{hemi.capitalize()} hemisphere  |  {region_group}',
                     color=title_col, fontsize=10, fontweight='bold')
        ax.tick_params(colors='#CCCCCC')
        for spine in ax.spines.values():
            spine.set_edgecolor('#333333')
        ax.legend(facecolor='#1A1A2E', edgecolor='#444444',
                  labelcolor='#CCCCCC', fontsize=7.5)

    fig.suptitle(
        f"RSA_r → {OUTCOME_LABELS[outcome]}  |  "
        f"{region_group}  |  Left vs Right",
        color=title_col, fontsize=12, fontweight='bold', y=1.01)

    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"    ✓ Interaction plot → {save_path.name}")


def plot_summary_heatmap(all_results: pd.DataFrame, save_path: Path):
    """
    Heatmap of RSA_r beta (Model1, combined) across all outcomes × region groups.
    Annotated with significance stars.
    """
    pivot_rows = []
    for outcome in OUTCOMES:
        for rg in REGION_GROUPS:
            sub = all_results[
                (all_results['outcome']      == outcome) &
                (all_results['region_group'] == rg) &
                (all_results['model']        == 'Model1') &
                (all_results['hemisphere']   == 'combined') &
                (all_results['col'].str.contains('RSA_r', na=False))
            ]
            if sub.empty:
                beta, stars = np.nan, ''
            else:
                r = sub.iloc[0]
                beta  = r['beta']
                stars = sig_stars(r.get('p_fdr', r['p_raw']))
            pivot_rows.append({
                'outcome':      OUTCOME_LABELS[outcome],
                'region_group': rg,
                'beta':         beta,
                'stars':        stars,
            })

    piv  = pd.DataFrame(pivot_rows)
    beta_mat  = piv.pivot(index='outcome', columns='region_group', values='beta')
    stars_mat = piv.pivot(index='outcome', columns='region_group', values='stars')
    beta_mat  = beta_mat[REGION_GROUPS]
    stars_mat = stars_mat[REGION_GROUPS]

    fig, ax = plt.subplots(figsize=(7, 3.5))
    fig.patch.set_facecolor('#0D1117')
    ax.set_facecolor('#0D1117')

    vmax = np.nanmax(np.abs(beta_mat.values))
    im = ax.imshow(beta_mat.values.astype(float),
                   aspect='auto', cmap='RdBu_r',
                   vmin=-vmax, vmax=vmax)

    for i in range(beta_mat.shape[0]):
        for j in range(beta_mat.shape[1]):
            val = beta_mat.values[i, j]
            star = stars_mat.values[i, j]
            if not np.isnan(val):
                txt = f"{val:.3f}\n{star}"
                ax.text(j, i, txt, ha='center', va='center',
                        color='white', fontsize=9, fontweight='bold')

    ax.set_xticks(range(len(REGION_GROUPS)))
    ax.set_xticklabels(REGION_GROUPS, color='#CCCCCC', fontsize=10)
    ax.set_yticks(range(len(beta_mat.index)))
    ax.set_yticklabels(beta_mat.index.tolist(), color='#CCCCCC', fontsize=10)

    cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label('β (RSA_r)', color='#CCCCCC', fontsize=9)
    cbar.ax.yaxis.set_tick_params(color='#CCCCCC')
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#CCCCCC')

    ax.set_title('RSA_r Effect Size (Model 1, combined) — '
                 'β across Outcomes × Region Groups',
                 color='#EEEEEE', fontsize=10, fontweight='bold', pad=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#444444')

    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  ✓ Summary heatmap → {save_path.name}")


def plot_beta_bars(all_results: pd.DataFrame, save_path: Path):
    """
    Grouped bar chart: RSA_r beta per region_group × hemisphere,
    separately for T_lag and SP_lag, using Model1 and Model4.
    """
    n_rows = len(OUTCOMES)
    fig, axes = plt.subplots(n_rows, 2, figsize=(14, 5 * n_rows))
    fig.patch.set_facecolor('#0D1117')
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    model_pairs = [('Model1', 'Bare'), ('Model4', 'Hemisphere interaction')]

    for row_idx, outcome in enumerate(OUTCOMES):
        for col_idx, (model_key, model_name) in enumerate(model_pairs):
            ax = axes[row_idx, col_idx]
            ax.set_facecolor('#111827')

            sub = all_results[
                (all_results['outcome'] == outcome) &
                (all_results['model']   == model_key) &
                (all_results['col'].str.contains('RSA_r', na=False)) &
                (~all_results['col'].str.contains(':', na=False))  # exclude interaction term itself
            ].copy()

            if sub.empty:
                ax.set_visible(False)
                continue

            x          = np.arange(len(REGION_GROUPS))
            width      = 0.25
            hemis_plot = ['left', 'right', 'combined']
            offsets    = [-width, 0, width]
            hemi_col   = {
                'left': HEMI_COLORS['left'],
                'right': HEMI_COLORS['right'],
                'combined': '#888888',
            }

            for hemi, offset in zip(hemis_plot, offsets):
                betas, errors, pvals = [], [], []
                for rg in REGION_GROUPS:
                    row = sub[
                        (sub['region_group'] == rg) &
                        (sub['hemisphere']   == hemi)
                    ]
                    if row.empty:
                        betas.append(0); errors.append(0); pvals.append(1.0)
                    else:
                        r = row.iloc[0]
                        betas.append(r['beta'])
                        errors.append(r['se'])
                        p = r.get('p_fdr', r['p_raw'])
                        pvals.append(p)

                bars = ax.bar(x + offset, betas, width,
                               color=hemi_col[hemi],
                               yerr=errors,
                               error_kw=dict(ecolor='#CCCCCC', capsize=3,
                                              elinewidth=1),
                               alpha=0.82, label=hemi.capitalize())

                # significance markers on bars
                for bar, p in zip(bars, pvals):
                    stars = sig_stars(p)
                    if stars:
                        h = bar.get_height()
                        sign = 1 if h >= 0 else -1
                        ax.text(bar.get_x() + bar.get_width() / 2,
                                h + sign * max(abs(h) * 0.05, 0.005),
                                stars, ha='center', va='bottom',
                                color='white', fontsize=8, fontweight='bold')

            ax.axhline(0, color='#666666', lw=1, ls='--')
            ax.set_xticks(x)
            ax.set_xticklabels(REGION_GROUPS, color='#CCCCCC', fontsize=9)
            ax.set_ylabel('β (RSA_r)', color='#CCCCCC', fontsize=9)
            ax.set_title(
                f"{OUTCOME_LABELS[outcome]}  ←  RSA_r\n"
                f"[{model_name}]",
                color='#EEEEEE', fontsize=9, fontweight='bold')
            ax.tick_params(colors='#CCCCCC')
            for spine in ax.spines.values():
                spine.set_edgecolor('#333333')
            ax.legend(facecolor='#1A1A2E', edgecolor='#444444',
                      labelcolor='#CCCCCC', fontsize=8)

    fig.suptitle('RSA_r → Temporal / Spatial Lag  |  β by Region Group × Hemisphere',
                 color='#EEEEEE', fontsize=13, fontweight='bold', y=1.01)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  ✓ Beta bar chart → {save_path.name}")


# ============================================================================
# MAIN ANALYSIS LOOP
# ============================================================================

def load_data() -> Optional[pd.DataFrame]:
    """Load and combine all allregions CSVs across experiments."""
    dfs = []
    for exp in EXPERIMENTS:
        for suffix in ['_allregions_rsa_lag.csv']:
            fpath = INPUT_DIR / f"ALL_SUBJECTS_{exp}{suffix}"
            if fpath.exists():
                tmp = pd.read_csv(fpath)
                tmp['experiment'] = exp
                dfs.append(tmp)
                print(f"  Loaded {fpath.name}  ({len(tmp):,} rows)")
                break
        else:
            # Fallback: try to reconstruct from per-region files
            region_dfs = []
            for region in ['LHP', 'RHP', 'LEC', 'REC', 'LPC', 'RPC']:
                rfpath = INPUT_DIR / f"ALL_SUBJECTS_{exp}_{region}_rsa_lag.csv"
                if rfpath.exists():
                    tmp = pd.read_csv(rfpath)
                    tmp['experiment'] = exp
                    region_dfs.append(tmp)
            if region_dfs:
                combined = pd.concat(region_dfs, ignore_index=True)
                dfs.append(combined)
                print(f"  Reconstructed {exp} from per-region files "
                      f"({len(combined):,} rows)")
            else:
                print(f"  ✗ No data found for {exp}")

    if not dfs:
        return None
    df = pd.concat(dfs, ignore_index=True)

    # Add region_group and hemisphere if not already present
    HEMISPHERE_MAP   = {r: ('left' if r.startswith('L') else 'right')
                        for r in ['LHP', 'RHP', 'LEC', 'REC', 'LPC', 'RPC']}
    REGION_GROUP_MAP = {
        'LHP': 'HP', 'RHP': 'HP',
        'LEC': 'EC', 'REC': 'EC',
        'LPC': 'PC', 'RPC': 'PC',
    }
    if 'hemisphere' not in df.columns:
        df['hemisphere']   = df['region'].map(HEMISPHERE_MAP)
    if 'region_group' not in df.columns:
        df['region_group'] = df['region'].map(REGION_GROUP_MAP)

    # Prefix subject IDs to avoid cross-experiment collisions
    df['subject'] = df['experiment'].astype(str) + '_' + df['subject'].astype(str)

    # Dummy-code hemisphere: left=0 (reference), right=1
    df['hemisphere_dummy'] = (df['hemisphere'] == 'right').astype(int)

    # Dummy-code experiment: DBOY1=0 (reference)
    ref_exp = 'DBOY1' if 'DBOY1' in df['experiment'].unique() else \
              df['experiment'].unique()[0]
    df['experiment_dummy'] = (df['experiment'] != ref_exp).astype(int)

    return df


def run_analysis_for_group(df: pd.DataFrame,
                            outcome: str,
                            region_group: str,
                            has_semsim: bool) -> Tuple[pd.DataFrame, str]:
    """
    Run Models 1–5 for a given outcome × region_group.
    Returns (all_rows_df, text_summary).
    """
    sub = df[df['region_group'] == region_group].copy()
    if sub.empty:
        return pd.DataFrame(), ""

    print(f"\n  ── {outcome}  |  {region_group}  ──────────────────────────")
    print(f"     Rows: {len(sub):,}  |  Subjects: {sub['subject'].nunique()}")

    all_rows = []
    text_blocks = [
        f"OUTCOME: {outcome}   REGION GROUP: {region_group}",
        f"N rows total: {len(sub):,}  |  subjects: {sub['subject'].nunique()}\n",
    ]

    def run_and_collect(pred_cols, model_key, model_desc,
                        pred_display, hemisphere='combined',
                        data_override=None):
        data = data_override if data_override is not None else sub
        res, _ = fit_lmm(data, outcome, pred_cols, model_key)
        rows = extract_rows(res, pred_display, model_key,
                            outcome, region_group, hemisphere)
        rows = apply_fdr_within_model(rows)
        if not rows.empty:
            all_rows.append(rows)
            block = format_block(f"{model_desc}  [{hemisphere}]", rows)
            text_blocks.append(block)
            print('\n' + block)
        return res

    # ── Models on COMBINED (all channels, hemisphere as predictor) ───────────

    # Model 1: bare
    run_and_collect(
        ['RSA_r', 'experiment_dummy'],
        'Model1',
        f'Model 1 — {outcome} ~ RSA_r + experiment',
        {'RSA_r': 'RSA_r', 'experiment_dummy': 'experiment'},
    )

    # Model 2: controlled
    ctrl_cols = ['RSA_r', 'output_lag', 'experiment_dummy']
    if has_semsim:
        ctrl_cols = ['RSA_r', 'output_lag', 'semantic_sim', 'experiment_dummy']
    run_and_collect(
        ctrl_cols,
        'Model2',
        f'Model 2 — {outcome} ~ RSA_r + controls',
        {c: c for c in ctrl_cols},
    )

    # Model 3: hemisphere main effect
    run_and_collect(
        ['RSA_r', 'hemisphere_dummy', 'experiment_dummy'],
        'Model3',
        f'Model 3 — {outcome} ~ RSA_r + hemisphere',
        {'RSA_r': 'RSA_r',
         'hemisphere_dummy': 'hemisphere (right vs left)',
         'experiment_dummy': 'experiment'},
    )

    # Model 4: RSA_r × hemisphere interaction
    run_and_collect(
        ['RSA_r', 'hemisphere_dummy', 'RSA_r:hemisphere_dummy',
         'experiment_dummy'],
        'Model4',
        f'Model 4 — {outcome} ~ RSA_r * hemisphere',
        {'RSA_r':                   'RSA_r (left hemisphere)',
         'hemisphere_dummy':        'hemisphere (right vs left)',
         'RSA_r:hemisphere_dummy':  'RSA_r × hemisphere',
         'experiment_dummy':        'experiment'},
    )

    # Model 5: full + hemisphere interaction
    full_int_cols = ['RSA_r', 'hemisphere_dummy', 'RSA_r:hemisphere_dummy',
                     'output_lag', 'experiment_dummy']
    full_int_disp = {
        'RSA_r':                  'RSA_r (left hemisphere)',
        'hemisphere_dummy':       'hemisphere (right vs left)',
        'RSA_r:hemisphere_dummy': 'RSA_r × hemisphere',
        'output_lag':             'output_lag',
        'experiment_dummy':       'experiment',
    }
    if has_semsim:
        full_int_cols = ['RSA_r', 'hemisphere_dummy', 'RSA_r:hemisphere_dummy',
                         'output_lag', 'semantic_sim', 'experiment_dummy']
        full_int_disp['semantic_sim'] = 'semantic_sim'
    run_and_collect(
        full_int_cols,
        'Model5',
        f'Model 5 — {outcome} ~ RSA_r * hemisphere + controls',
        full_int_disp,
    )

    # ── Models run SEPARATELY per hemisphere (Models 1 & 2) ─────────────────
    for hemi in ['left', 'right']:
        hemi_data = sub[sub['hemisphere'] == hemi].copy()
        if len(hemi_data) < 20:
            continue

        run_and_collect(
            ['RSA_r', 'experiment_dummy'],
            'Model1',
            f'Model 1 — {outcome} ~ RSA_r + experiment',
            {'RSA_r': 'RSA_r', 'experiment_dummy': 'experiment'},
            hemisphere=hemi,
            data_override=hemi_data,
        )

        ctrl_h = ['RSA_r', 'output_lag', 'experiment_dummy']
        if has_semsim:
            ctrl_h = ['RSA_r', 'output_lag', 'semantic_sim', 'experiment_dummy']
        run_and_collect(
            ctrl_h,
            'Model2',
            f'Model 2 — {outcome} ~ RSA_r + controls',
            {c: c for c in ctrl_h},
            hemisphere=hemi,
            data_override=hemi_data,
        )

    result_df = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
    return result_df, '\n\n'.join(text_blocks)


# ============================================================================
# ENTRY POINT
# ============================================================================

def main():
    print(f"\n{'='*80}")
    print("RSA_r → T_lag / SP_lag  |  All region groups × hemispheres")
    print(f"{'='*80}")

    df = load_data()
    if df is None or df.empty:
        print("No data loaded. Exiting.")
        return

    has_semsim = ('semantic_sim' in df.columns and
                  df['semantic_sim'].notna().any())
    print(f"\n  semantic_sim available: {has_semsim}")
    print(f"  Outcomes     : {OUTCOMES}")
    print(f"  Region groups: {REGION_GROUPS}")
    print(f"  N total rows : {len(df):,}")
    print(f"  N subjects   : {df['subject'].nunique()}")

    ALL_RESULTS  = []
    ALL_TEXT     = {}

    for outcome in OUTCOMES:
        for region_group in REGION_GROUPS:
            res_df, text = run_analysis_for_group(
                df, outcome, region_group, has_semsim)

            if not res_df.empty:
                ALL_RESULTS.append(res_df)

            # Save per outcome × region group
            tag      = f"{outcome}_{region_group}"
            csv_path = OUTPUT_DIR / f"LMM_{tag}_results.csv"
            txt_path = OUTPUT_DIR / f"LMM_{tag}_results.txt"
            if not res_df.empty:
                res_df.to_csv(csv_path, index=False)
                print(f"    ✓ Saved: {csv_path.name}")
            if text:
                with open(txt_path, 'w') as f:
                    f.write(text)
                print(f"    ✓ Saved: {txt_path.name}")

            ALL_TEXT[tag] = text

    # ── Combine all results ───────────────────────────────────────────────────
    if not ALL_RESULTS:
        print("  No results generated.")
        return

    master_df = pd.concat(ALL_RESULTS, ignore_index=True)
    master_df.to_csv(OUTPUT_DIR / 'LMM_ALL_results.csv', index=False)
    print(f"\n  ✓ Master results → LMM_ALL_results.csv  ({len(master_df):,} rows)")

    # ── Plots ─────────────────────────────────────────────────────────────────
    print("\n  Generating plots …")

    for outcome in OUTCOMES:
        for region_group in REGION_GROUPS:
            sub_results = master_df[
                (master_df['outcome']      == outcome) &
                (master_df['region_group'] == region_group)
            ]
            if sub_results.empty:
                continue

            # Forest plot
            plot_forest(
                master_df, outcome, region_group,
                PLOT_DIR / f"forest_{outcome}_{region_group}.png")

            # Interaction scatter plot
            plot_interaction(
                master_df, outcome, region_group, df,
                PLOT_DIR / f"interaction_{outcome}_{region_group}.png")

    # Summary heatmap
    plot_summary_heatmap(
        master_df,
        PLOT_DIR / 'summary_heatmap.png')

    # Beta bar chart
    plot_beta_bars(
        master_df,
        PLOT_DIR / 'beta_bars_all.png')

    print(f"\n{'='*80}")
    print("✓ ANALYSIS COMPLETE")
    print(f"  Results : {OUTPUT_DIR}")
    print(f"  Plots   : {PLOT_DIR}")
    print(f"{'='*80}")


if __name__ == '__main__':
    main()


RSA_r → T_lag / SP_lag  |  All region groups × hemispheres
  Loaded ALL_SUBJECTS_DBOY1_allregions_rsa_lag.csv  (5,330 rows)
  Loaded ALL_SUBJECTS_EFRCourierReadOnly_allregions_rsa_lag.csv  (749 rows)
  Loaded ALL_SUBJECTS_EFRCourierOpenLoop_allregions_rsa_lag.csv  (514 rows)

  semantic_sim available: True
  Outcomes     : ['T_lag', 'SP_lag']
  Region groups: ['HP', 'EC', 'PC']
  N total rows : 6,593
  N subjects   : 46

  ── T_lag  |  HP  ──────────────────────────
     Rows: 4,004  |  Subjects: 44
    [Model1] T_lag ~ RSA_r + experiment_dummy  |  N=4,004
    [Model1] optimizer=lbfgs  converged=True  llf=-10156.3877  AIC=nan

Model 1 — T_lag ~ RSA_r + experiment  [combined]
------------------------------------------------------------------------------------------
Predictor                                     β       SE        z      p_raw      p_fdr        AIC        N   sig
------------------------------------------------------------------------------------------
RSA_r              

KeyError: "['RSA_r:hemisphere_dummy'] not in index"

In [4]:
#!/usr/bin/env python3
"""
LMM Analysis: T_lag / SP_lag → RSA_r
Hippocampus only (LHP / RHP), stratified by FREQUENCY BAND
----------------------------------------------------------------------
RSA_r is the OUTCOME.  T_lag and SP_lag are the PREDICTORS OF INTEREST,
tested in separate model sets.

Z-SCORING (within-subject, within-band):
  Before fitting any model, for each band subset:
    - RSA_r      is z-scored within each subject  → RSA_r_z
    - T_lag      is z-scored within each subject  → T_lag_z
    - SP_lag     is z-scored within each subject  → SP_lag_z
    - output_lag is z-scored within each subject  → output_lag_z
  Models are fit on the z-scored columns.  Betas are therefore in SD units,
  directly comparable across bands and predictors.

Input : ./rsa_lag_hc_bands/ALL_SUBJECTS_{exp}_hc_allbands_rsa_lag.csv
        (output of rsa_lag_hc_bands.py)

For each PREDICTOR in [T_lag, SP_lag]:
  For each BAND in [theta, alpha, beta, gamma]:

    Model 1 (bare):
        RSA_r ~ predictor + experiment_dummy              + (1|subj/sess)

    Model 2 (controlled):
        RSA_r ~ predictor + output_lag + semantic_sim
                          + experiment_dummy              + (1|subj/sess)

    Model 3 (hemisphere main effect):
        RSA_r ~ predictor + hemisphere_dummy
                          + experiment_dummy              + (1|subj/sess)

    Model 4 (predictor × hemisphere interaction):
        RSA_r ~ predictor * hemisphere_dummy
                          + experiment_dummy              + (1|subj/sess)

    Model 5 (full + hemisphere interaction):
        RSA_r ~ predictor * hemisphere_dummy + output_lag
                          + semantic_sim + experiment_dummy
                                                          + (1|subj/sess)

    Models 1 & 2 are also run SEPARATELY for left (LHP) and right (RHP).

hemisphere_dummy : left=0 (reference), right=1
experiment_dummy : DBOY1=0 (reference), others=1

Outputs:
  ./rsa_lag_hc_bands/LMM_{predictor}_{band}_results.csv
  ./rsa_lag_hc_bands/LMM_{predictor}_{band}_results.txt
  ./rsa_lag_hc_bands/LMM_ALL_results.csv
  ./rsa_lag_hc_bands/plots/forest_{predictor}_{band}.png
  ./rsa_lag_hc_bands/plots/interaction_{predictor}_{band}.png
  ./rsa_lag_hc_bands/plots/summary_heatmap.png
  ./rsa_lag_hc_bands/plots/beta_bars_all.png
"""

import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import pearsonr
from statsmodels.regression.mixed_linear_model import MixedLM
from statsmodels.stats.multitest import fdrcorrection

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

EXPERIMENTS = ['DBOY1', 'EFRCourierReadOnly', 'EFRCourierOpenLoop']
INPUT_DIR   = Path('./rsa_lag_hc_bands')
OUTPUT_DIR  = Path('./rsa_lag_hc_bands')
PLOT_DIR    = OUTPUT_DIR / 'plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

OUTCOME    = 'RSA_r'

PREDICTORS = ['T_lag', 'SP_lag']
BANDS      = ['theta', 'alpha', 'beta', 'gamma']   # primary loop axis (replaces REGION_GROUPS)

# ---- Palette (light background) ---------------------------------------------
BG_COLOR    = 'white'
AX_COLOR    = '#F7F7F7'
GRID_COLOR  = '#DDDDDD'
TEXT_COLOR  = '#222222'
SPINE_COLOR = '#AAAAAA'

HEMI_COLORS = {
    'left':     '#1A3A6B',   # deep navy  (LHP convention)
    'right':    '#8B1A1A',   # deep crimson (RHP convention)
    'combined': '#666666',
}

BAND_COLORS = {
    'theta': '#4575B4',
    'alpha': '#74ADD1',
    'beta':  '#F46D43',
    'gamma': '#D73027',
}

PRED_LABELS = {
    'T_lag':  'Temporal Lag (T_lag)',
    'SP_lag': 'Spatial Lag (SP_lag)',
}
CROSS_COVARIATE = {
    'T_lag':  'SP_lag',
    'SP_lag': 'T_lag',
}
MODEL_LABELS = {
    'Model1': 'Bare',
    'Model2': 'Controlled',
    'Model3': 'Hemi main',
    'Model4': 'Hemi interaction',
    'Model5': 'Full + interaction',
}

# ============================================================================
# LMM FITTING
# ============================================================================

def fit_lmm(df: pd.DataFrame,
            pred_cols: List[str],
            label: str,
            formula_rhs: Optional[str] = None) -> Tuple[Optional[object], int]:
    """
    Fit RSA_r ~ formula_rhs with nested random effects (1|subject/session).

    pred_cols   : real DataFrame column names used for dropna
    formula_rhs : explicit RHS; if None, built as ' + '.join(pred_cols).
    """
    df = df.copy()
    df['subj_sess'] = df['subject'].astype(str) + '_' + df['session'].astype(str)

    real_cols = [c for c in pred_cols if c in df.columns]
    keep = [OUTCOME] + real_cols + ['subject', 'subj_sess']
    df   = df[keep].dropna()

    if len(df) < 20:
        print(f"    [{label}] Too few rows ({len(df)}) — skipping")
        return None, 0

    rhs     = formula_rhs if formula_rhs else ' + '.join(pred_cols)
    formula = f"{OUTCOME} ~ {rhs}"
    print(f"    [{label}] {formula}  |  N={len(df):,}")

    model = MixedLM.from_formula(
        formula,
        data       = df,
        groups     = df['subject'],
        vc_formula = {'subj_sess': '0 + C(subj_sess)'},
    )

    result = None
    for method in ['lbfgs', 'nm', 'powell']:
        try:
            result = model.fit(reml=True, method=method)
            if np.isfinite(result.llf):
                print(f"    [{label}] optimizer={method}  "
                      f"converged={getattr(result, 'converged', None)}  "
                      f"llf={result.llf:.4f}  AIC={result.aic:.4f}")
                break
            else:
                print(f"    [{label}] llf=NaN with {method}, trying next …")
        except Exception as e:
            print(f"    [{label}] {method} failed: {e}")
            result = None

    if result is None or not np.isfinite(result.llf):
        print(f"    [{label}] WARNING: fit unsuccessful.")
    return result, len(df)


# ============================================================================
# RESULT EXTRACTION
# ============================================================================

def extract_rows(result,
                 pred_display: Dict[str, str],
                 model_label: str,
                 predictor: str,
                 band: str,
                 hemisphere: str = 'combined') -> pd.DataFrame:
    if result is None:
        return pd.DataFrame()
    rows = []
    for col, name in pred_display.items():
        matched = col if col in result.params.index else None
        if matched is None:
            hits = [k for k in result.params.index
                    if col.lower() in k.lower()]
            matched = hits[0] if hits else None
        if matched is None:
            print(f"    WARNING: '{col}' not found in params — skipping")
            continue
        rows.append({
            'predictor_of_interest': predictor,
            'band':                  band,          # ← band replaces region_group
            'hemisphere':            hemisphere,
            'model':                 model_label,
            'term':                  name,
            'col':                   col,
            'beta':                  result.params[matched],
            'se':                    result.bse[matched],
            'z':                     result.tvalues[matched],
            'p_raw':                 result.pvalues[matched],
            'llf':                   result.llf,
            'aic':                   result.aic,
            'nobs':                  int(result.nobs),
        })
    return pd.DataFrame(rows)


def apply_fdr_within_model(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    _, df['p_fdr'] = fdrcorrection(df['p_raw'].values)
    return df


# ============================================================================
# TEXT FORMATTING
# ============================================================================

def sig_stars(p: float) -> str:
    return ('***' if p < 0.001 else
            '**'  if p < 0.01  else
            '*'   if p < 0.05  else
            '†'   if p < 0.10  else '')


def format_block(title: str, rows_df: pd.DataFrame) -> str:
    sep  = '=' * 92
    sep2 = '-' * 92
    hdr  = (f"{'Term':<40} {'β':>8} {'SE':>8} {'z':>8} "
            f"{'p_raw':>10} {'p_fdr':>10} {'AIC':>10} {'N':>8} {'sig':>5}")
    lines = [sep, title, sep2, hdr, sep2]
    for _, row in rows_df.iterrows():
        aic_s = f"{row['aic']:>10.2f}" if pd.notna(row.get('aic')) else '       NaN'
        p_fdr = row.get('p_fdr', np.nan)
        lines.append(
            f"{row['term']:<40} {row['beta']:>8.4f} {row['se']:>8.4f} "
            f"{row['z']:>8.3f} {row['p_raw']:>10.4f} "
            f"{p_fdr:>10.4f} {aic_s} {int(row['nobs']):>8,} "
            f"{sig_stars(p_fdr):>5}"
        )
    lines += [sep2,
              'FDR: BH correction within each model across terms.',
              '† p<.10  * p<.05  ** p<.01  *** p<.001',
              'Outcome = RSA_r_z (z-scored within subject).',
              'Predictors T_lag_z / SP_lag_z / output_lag_z = z-scored within subject.',
              'hemisphere_dummy: LHP=0 (ref), RHP=1.',
              'experiment_dummy: DBOY1=0 (ref).',
              sep]
    return '\n'.join(lines)


# ============================================================================
# PLOTTING
# ============================================================================

def _style_ax(ax):
    ax.set_facecolor(AX_COLOR)
    ax.tick_params(colors=TEXT_COLOR, labelsize=8.5)
    ax.xaxis.label.set_color(TEXT_COLOR)
    ax.yaxis.label.set_color(TEXT_COLOR)
    ax.title.set_color(TEXT_COLOR)
    ax.grid(True, color=GRID_COLOR, lw=0.6, zorder=0)
    for spine in ax.spines.values():
        spine.set_edgecolor(SPINE_COLOR)
        spine.set_linewidth(0.8)


def plot_forest(all_results: pd.DataFrame,
                predictor: str,
                band: str,
                save_path: Path):
    """
    Forest plot: beta of the predictor of interest ± 95%CI across
    models and hemisphere subsets, for one band.
    """
    df = all_results[
        (all_results['predictor_of_interest'] == predictor) &
        (all_results['band']                  == band) &
        (all_results['col']                   == predictor)
    ].copy()

    if df.empty:
        print(f"    No rows for forest plot {predictor} {band}")
        return

    df['ci_lo'] = df['beta'] - 1.96 * df['se']
    df['ci_hi'] = df['beta'] + 1.96 * df['se']

    models  = [m for m in ['Model1', 'Model2', 'Model3', 'Model4', 'Model5']
               if m in df['model'].values]
    hemis   = ['left', 'right', 'combined']
    markers = {'left': 'o', 'right': 's', 'combined': 'D'}

    fig, ax = plt.subplots(figsize=(10, max(4, len(models) * 1.6 + 1)))
    fig.patch.set_facecolor(BG_COLOR)
    _style_ax(ax)

    y_pos, yticks, ylabels = 0, [], []

    for model in models:
        sub = df[df['model'] == model]
        for hemi in hemis:
            row = sub[sub['hemisphere'] == hemi]
            if row.empty:
                continue
            row  = row.iloc[0]
            col  = HEMI_COLORS[hemi]
            xerr = [[row['beta'] - row['ci_lo']], [row['ci_hi'] - row['beta']]]
            ax.errorbar(row['beta'], y_pos, xerr=xerr,
                        fmt=markers[hemi], color=col, ecolor=col,
                        elinewidth=1.5, capsize=4, capthick=1.5,
                        markersize=7, zorder=3)
            p_show = row.get('p_fdr', row['p_raw'])
            stars  = sig_stars(p_show)
            if stars:
                offset = abs(row['ci_hi'] - row['beta']) * 0.15 + 0.001
                ax.text(row['ci_hi'] + offset, y_pos, stars,
                        color=col, va='center', fontsize=9, fontweight='bold')
            yticks.append(y_pos)
            ylabels.append(f"{MODEL_LABELS.get(model, model)}  [{hemi}]")
            y_pos -= 1
        y_pos -= 0.5

    ax.axvline(0, color=SPINE_COLOR, lw=1.2, ls='--', zorder=1)
    ax.set_yticks(yticks)
    ax.set_yticklabels(ylabels, fontsize=8.5)
    ax.set_xlabel(f'β  ({PRED_LABELS[predictor]})', fontsize=10)

    legend_handles = [
        Line2D([0], [0], marker='o', color='w',
               markerfacecolor=HEMI_COLORS['left'],     markersize=8, label='LHP (left)'),
        Line2D([0], [0], marker='s', color='w',
               markerfacecolor=HEMI_COLORS['right'],    markersize=8, label='RHP (right)'),
        Line2D([0], [0], marker='D', color='w',
               markerfacecolor=HEMI_COLORS['combined'], markersize=8, label='Combined'),
    ]
    ax.legend(handles=legend_handles, loc='upper right',
              facecolor='white', edgecolor=SPINE_COLOR, fontsize=8)
    ax.set_title(f"RSA_r ~ {PRED_LABELS[predictor]}  |  {band.capitalize()} band",
                 fontsize=12, fontweight='bold', pad=10)

    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"    ✓ Forest plot → {save_path.name}")


def _subject_slopes(hd: pd.DataFrame, predictor: str) -> pd.DataFrame:
    """
    Compute one OLS slope of RSA_r_z ~ predictor_z per subject.
    Both columns are expected to already be z-scored before calling this.
    """
    outcome_z = 'RSA_r_z'
    rows = []
    for subj, sg in hd.groupby('subject'):
        sg = sg.dropna(subset=[predictor, outcome_z])
        if len(sg) < 3:
            continue
        x = sg[predictor].values.astype(float)
        y = sg[outcome_z].values.astype(float)
        if x.std() == 0:
            continue
        m, b = np.polyfit(x, y, 1)
        rows.append({'subject': subj, 'slope': m,
                     'intercept': b, 'n_pairs': len(sg)})
    return pd.DataFrame(rows)


def _subject_zscore_rsa(hd: pd.DataFrame) -> pd.DataFrame:
    hd = hd.copy()
    def zscore(x):
        s = x.std()
        return (x - x.mean()) / s if s > 0 else x - x.mean()
    hd['RSA_r_z'] = hd.groupby('subject')[OUTCOME].transform(zscore)
    return hd


def zscore_within_subject(df: pd.DataFrame,
                           cols: List[str]) -> pd.DataFrame:
    """
    Z-score each column in `cols` within each subject and store the result
    in a new column named  <col>_z.  The original columns are preserved.

    Subjects with zero variance on a column get a column of 0s (not NaN)
    so they are not silently dropped by downstream dropna().
    """
    df = df.copy()

    def _z(x: pd.Series) -> pd.Series:
        s = x.std(ddof=1)
        return (x - x.mean()) / s if s > 0 else pd.Series(0.0, index=x.index)

    for col in cols:
        if col not in df.columns:
            continue
        df[f'{col}_z'] = df.groupby('subject')[col].transform(_z)

    return df


def plot_interaction(all_results: pd.DataFrame,
                     predictor: str,
                     band: str,
                     raw_df: pd.DataFrame,
                     save_path: Path):
    """
    Two-panel figure per hemisphere (LHP / RHP), each panel has two subplots:

    TOP  — spaghetti + group mean (RSA_r_z within subject)
    BOTTOM — subject slope dot plot (OLS slope per subject)

    Data is z-scored within subject (same as LMM) before plotting,
    so axes are in SD units and match the reported betas.
    """
    # ── Subset to this band, then z-score exactly as in run_analysis_for_band ─
    sub_df = raw_df[raw_df['band'] == band].copy()
    sub_df = sub_df.dropna(subset=[predictor, OUTCOME])
    if sub_df.empty:
        return

    Z_COLS = ['RSA_r', 'T_lag', 'SP_lag', 'output_lag']
    sub_df = zscore_within_subject(sub_df, Z_COLS)

    pred_z     = f'{predictor}_z'
    is_discrete = (predictor == 'T_lag')
    hemis       = ['left', 'right']

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.patch.set_facecolor(BG_COLOR)

    for col_idx, hemi in enumerate(hemis):
        hd     = sub_df[sub_df['hemisphere'] == hemi].copy()
        ax_sp  = axes[0, col_idx]
        ax_sl  = axes[1, col_idx]
        _style_ax(ax_sp)
        _style_ax(ax_sl)

        hemi_label = 'LHP' if hemi == 'left' else 'RHP'

        if hd.empty:
            ax_sp.set_visible(False)
            ax_sl.set_visible(False)
            continue

        color   = HEMI_COLORS[hemi]
        n_subj  = hd['subject'].nunique()
        n_pairs = len(hd)

        # Group-level OLS annotation — on z-scored columns
        x_all = hd[pred_z].values.astype(float)
        y_all = hd['RSA_r_z'].values.astype(float)
        valid_mask = np.isfinite(x_all) & np.isfinite(y_all)
        if valid_mask.sum() >= 3:
            slope_grp, _ = np.polyfit(x_all[valid_mask], y_all[valid_mask], 1)
            r_val, p_val = pearsonr(x_all[valid_mask], y_all[valid_mask])
        else:
            slope_grp, r_val, p_val = 0.0, 0.0, 1.0
        p_str = f'p={p_val:.3f}' if p_val >= 0.001 else 'p<0.001'

        if is_discrete:
            MAX_VALS = 20
            top_vals = sorted(
                hd[pred_z].value_counts()
                .nlargest(MAX_VALS).index.tolist())
            x_grid = np.array(top_vals, dtype=float)
        else:
            N_BINS = 12
            hd['_bin'] = pd.cut(hd[pred_z], bins=N_BINS)
            bin_mids   = (hd.groupby('_bin', observed=True)[pred_z]
                           .mean().dropna())
            x_grid     = bin_mids.values

        # ── TOP: Spaghetti ────────────────────────────────────────────────────
        subj_lines = []

        for subj, sg in hd.groupby('subject'):
            sg = sg.dropna(subset=[pred_z, 'RSA_r_z'])
            if len(sg) < 3:
                continue
            if is_discrete:
                pts = (sg.groupby(pred_z)['RSA_r_z']
                         .mean()
                         .reindex(top_vals))
            else:
                pts = (sg.groupby('_bin', observed=True)['RSA_r_z']
                         .mean())
                pts.index = pts.index.map(
                    lambda b: b.mid if hasattr(b, 'mid') else np.nan)
                pts = pts.dropna()

            pts = pts.dropna()
            if len(pts) < 2:
                continue

            subj_lines.append(pts.values)
            ax_sp.plot(x_grid[:len(pts.values)], pts.values,
                       color=color, alpha=0.15, lw=0.9, zorder=2)

        if subj_lines:
            max_len  = max(len(s) for s in subj_lines)
            padded   = np.array([
                np.pad(s.astype(float),
                       (0, max_len - len(s)),
                       constant_values=np.nan)
                for s in subj_lines
            ])
            grp_mean = np.nanmean(padded, axis=0)
            grp_sem  = (np.nanstd(padded, axis=0) /
                        np.sqrt((~np.isnan(padded)).sum(axis=0)))
            xg = x_grid[:max_len]

            ax_sp.fill_between(xg,
                               grp_mean - grp_sem,
                               grp_mean + grp_sem,
                               color=color, alpha=0.20, zorder=3)
            ax_sp.plot(xg, grp_mean,
                       color=color, lw=2.5, zorder=4,
                       label=f'Group mean ± SEM  (n={n_subj} subj)')

            valid = np.isfinite(grp_mean) & np.isfinite(xg)
            if valid.sum() >= 2:
                m_z, b_z = np.polyfit(xg[valid], grp_mean[valid], 1)
                ax_sp.plot(xg[valid],
                           m_z * xg[valid] + b_z,
                           color=TEXT_COLOR, lw=1.5, ls='--',
                           alpha=0.6, zorder=5,
                           label=f'OLS  r={r_val:.3f} {p_str}')

        ax_sp.axhline(0, color=SPINE_COLOR, lw=1, ls=':', zorder=1)
        if is_discrete:
            ax_sp.set_xticks(x_grid)
            ax_sp.set_xticklabels(
                [str(int(v)) for v in x_grid], fontsize=7,
                rotation=45 if len(x_grid) > 12 else 0)
        ax_sp.set_xlabel(f'{PRED_LABELS[predictor]}  [z-scored]', fontsize=9)
        ax_sp.set_ylabel('RSA_r_z  (z-scored within subject)', fontsize=9)
        ax_sp.set_title(
            f'{hemi_label}  |  {band.capitalize()} band\n'
            f'Spaghetti: each line = 1 subject  '
            f'(n={n_subj}, {n_pairs:,} pairs)',
            fontsize=10, fontweight='bold')
        ax_sp.legend(facecolor='white', edgecolor=SPINE_COLOR, fontsize=8)

        # ── BOTTOM: Subject slope dot plot ────────────────────────────────────
        slopes_df = _subject_slopes(hd, pred_z)   # ← use z-scored predictor

        if slopes_df.empty:
            ax_sl.set_visible(False)
            continue

        slopes     = slopes_df['slope'].values
        n_sl       = len(slopes)
        mean_slope = slopes.mean()
        sem_slope  = slopes.std() / np.sqrt(n_sl)
        ci95       = 1.96 * sem_slope
        n_neg      = (slopes < 0).sum()
        n_pos      = (slopes > 0).sum()

        order      = np.argsort(slopes)
        y_pos_sl   = np.arange(n_sl)

        dot_colors = [HEMI_COLORS['left'] if s < 0
                      else HEMI_COLORS['right']
                      for s in slopes[order]]

        ax_sl.scatter(slopes[order], y_pos_sl,
                      c=dot_colors, s=40, zorder=3,
                      edgecolors='white', linewidths=0.4)

        for yi, si, dc in zip(y_pos_sl, slopes[order], dot_colors):
            ax_sl.plot([0, si], [yi, yi],
                       color=dc, alpha=0.35, lw=0.8, zorder=2)

        ax_sl.axvline(0, color=SPINE_COLOR, lw=1.2, ls='--', zorder=1)
        ax_sl.axvspan(mean_slope - ci95, mean_slope + ci95,
                      color=color, alpha=0.12, zorder=0)
        ax_sl.axvline(mean_slope, color=color, lw=2.5, zorder=4,
                      label=f'Mean β={mean_slope:.4f} ± {ci95:.4f} (95%CI)')

        from scipy.stats import ttest_1samp
        t_stat, p_1samp = ttest_1samp(slopes, 0)
        p1_str = f'p={p_1samp:.3f}' if p_1samp >= 0.001 else 'p<0.001'
        stars  = sig_stars(p_1samp)
        ax_sl.text(mean_slope, n_sl + 0.3,
                   f'{stars}  {p1_str}' if stars else p1_str,
                   ha='center', va='bottom',
                   color=color, fontsize=9, fontweight='bold')

        ax_sl.set_yticks([])
        ax_sl.set_xlabel(
            f'Subject-level OLS slope  (RSA_r_z ~ {pred_z})', fontsize=9)
        ax_sl.set_title(
            f'{hemi_label}  |  {band.capitalize()} band\n'
            f'Each dot = 1 subject slope  '
            f'({n_neg}/{n_sl} negative, {n_pos}/{n_sl} positive)',
            fontsize=10, fontweight='bold')
        ax_sl.legend(facecolor='white', edgecolor=SPINE_COLOR, fontsize=8)

        pct_neg = n_neg / n_sl * 100
        ax_sl.text(0.02, 0.04,
                   f'{pct_neg:.0f}% of subjects show negative slope',
                   transform=ax_sl.transAxes,
                   fontsize=8, color=TEXT_COLOR, style='italic')

    fig.suptitle(
        f"RSA_r ~ {PRED_LABELS[predictor]}  |  {band.capitalize()} band\n"
        f"Top: z-scored spaghetti + group mean  |  "
        f"Bottom: per-subject slopes",
        fontsize=12, fontweight='bold', y=1.01, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"    ✓ Interaction plot → {save_path.name}")


def plot_summary_heatmap(all_results: pd.DataFrame, save_path: Path):
    """
    Heatmap of predictor beta (Model1, combined) across
    predictor × band. Annotated with significance stars.
    """
    pivot_rows = []
    for pred in PREDICTORS:
        for band in BANDS:
            sub = all_results[
                (all_results['predictor_of_interest'] == pred) &
                (all_results['band']                  == band) &
                (all_results['model']                 == 'Model1') &
                (all_results['hemisphere']            == 'combined') &
                (all_results['col']                   == pred)
            ]
            if sub.empty:
                beta, stars = np.nan, ''
            else:
                r     = sub.iloc[0]
                beta  = r['beta']
                stars = sig_stars(r.get('p_fdr', r['p_raw']))
            pivot_rows.append({
                'predictor': PRED_LABELS[pred],
                'band':      band,
                'beta':      beta,
                'stars':     stars,
            })

    piv       = pd.DataFrame(pivot_rows)
    beta_mat  = piv.pivot(index='predictor', columns='band', values='beta')
    stars_mat = piv.pivot(index='predictor', columns='band', values='stars')
    beta_mat  = beta_mat[BANDS]
    stars_mat = stars_mat[BANDS]

    fig, ax = plt.subplots(figsize=(9, 3.2))
    fig.patch.set_facecolor(BG_COLOR)

    vmax = np.nanmax(np.abs(beta_mat.values)) or 1.0
    im   = ax.imshow(beta_mat.values.astype(float),
                     aspect='auto', cmap='RdBu_r',
                     vmin=-vmax, vmax=vmax)

    for i in range(beta_mat.shape[0]):
        for j in range(beta_mat.shape[1]):
            val  = beta_mat.values[i, j]
            star = stars_mat.values[i, j]
            if not np.isnan(val):
                cell_norm = (val + vmax) / (2 * vmax)
                txt_color = ('white' if (cell_norm < 0.35 or cell_norm > 0.65)
                             else TEXT_COLOR)
                ax.text(j, i, f"{val:.3f}\n{star}",
                        ha='center', va='center',
                        color=txt_color, fontsize=9, fontweight='bold')

    ax.set_xticks(range(len(BANDS)))
    ax.set_xticklabels([b.capitalize() for b in BANDS], fontsize=10)
    ax.set_yticks(range(len(beta_mat.index)))
    ax.set_yticklabels(beta_mat.index.tolist(), fontsize=9)
    ax.tick_params(colors=TEXT_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(SPINE_COLOR)

    cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label('β  (predictor → RSA_r)', fontsize=9, color=TEXT_COLOR)
    cbar.ax.yaxis.set_tick_params(color=TEXT_COLOR)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_COLOR)

    ax.set_title('RSA_r ~ predictor  |  β (Model 1, combined)  '
                 'across Predictors × Frequency Bands',
                 fontsize=10, fontweight='bold', pad=8, color=TEXT_COLOR)

    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"  ✓ Summary heatmap → {save_path.name}")


def plot_beta_bars(all_results: pd.DataFrame, save_path: Path):
    """
    Two-row bar chart per predictor:

    Row 1 — Model 1 (bare):  three bars per band
            LHP   = beta from Model1 on left-hemisphere data only
            RHP   = beta from Model1 on right-hemisphere data only
            Combined = beta from Model1 on pooled data

    Row 2 — Model 4 (hemisphere interaction): two bars per band
            Main effect  = beta of predictor (= LHP slope, reference)
            Interaction  = beta of predictor × hemisphere_dummy
                           (= difference in slope: RHP minus LHP)
    """
    n_cols = len(PREDICTORS)
    n_rows = 2
    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(7 * n_cols, 5 * n_rows))
    fig.patch.set_facecolor(BG_COLOR)
    if n_cols == 1:
        axes = axes[:, np.newaxis]

    x     = np.arange(len(BANDS))
    width = 0.25

    for col_idx, pred in enumerate(PREDICTORS):
        int_term = f'{pred}:hemisphere_dummy'

        # ── Row 0: Model 1 — LHP / RHP / Combined ────────────────────────────
        ax0 = axes[0, col_idx]
        _style_ax(ax0)

        hemis_plot = ['left', 'right', 'combined']
        hemi_labels = ['LHP', 'RHP', 'Combined']
        offsets    = [-width, 0, width]

        for hemi, hlabel, offset in zip(hemis_plot, hemi_labels, offsets):
            betas, errors, pvals = [], [], []
            for band in BANDS:
                row = all_results[
                    (all_results['predictor_of_interest'] == pred) &
                    (all_results['model']                 == 'Model1') &
                    (all_results['hemisphere']            == hemi) &
                    (all_results['band']                  == band) &
                    (all_results['col']                   == pred)
                ]
                if row.empty:
                    betas.append(np.nan); errors.append(0); pvals.append(1.0)
                else:
                    r = row.iloc[0]
                    betas.append(r['beta'])
                    errors.append(r['se'])
                    pvals.append(r.get('p_fdr', r['p_raw']))

            plot_betas  = [b if np.isfinite(b) else 0 for b in betas]
            plot_errors = [e if np.isfinite(betas[i]) else 0
                           for i, e in enumerate(errors)]

            bars = ax0.bar(x + offset, plot_betas, width,
                           color=HEMI_COLORS[hemi],
                           yerr=plot_errors,
                           error_kw=dict(ecolor=TEXT_COLOR, capsize=3,
                                          elinewidth=1),
                           alpha=0.78,
                           label=hlabel)

            for bar, beta, p in zip(bars, betas, pvals):
                if not np.isfinite(beta):
                    continue
                stars = sig_stars(p)
                if stars:
                    h    = bar.get_height()
                    sign = 1 if h >= 0 else -1
                    ax0.text(bar.get_x() + bar.get_width() / 2,
                             h + sign * max(abs(h) * 0.05, 0.005),
                             stars, ha='center', va='bottom',
                             color=TEXT_COLOR, fontsize=9, fontweight='bold')

        ax0.axhline(0, color=SPINE_COLOR, lw=1.0, ls='--')
        ax0.set_xticks(x)
        ax0.set_xticklabels([b.capitalize() for b in BANDS], fontsize=9)
        ax0.set_xlabel('Frequency Band', fontsize=9)
        ax0.set_ylabel('β', fontsize=9)
        ax0.set_title(
            f"RSA_r ~ {PRED_LABELS[pred]}\n"
            f"[Model 1 bare — LHP / RHP / Combined]",
            fontsize=9, fontweight='bold')
        ax0.legend(facecolor='white', edgecolor=SPINE_COLOR, fontsize=8)

        # ── Row 1: Model 4 — main effect (LHP) + interaction ─────────────────
        ax1 = axes[1, col_idx]
        _style_ax(ax1)

        terms = [
            (pred,     'Main effect\n(LHP slope)',   HEMI_COLORS['left'],  -width/2),
            (int_term, 'Interaction\n(RHP − LHP)',   HEMI_COLORS['right'],  width/2),
        ]

        for col_key, term_label, color, offset in terms:
            betas, errors, pvals = [], [], []
            for band in BANDS:
                row = all_results[
                    (all_results['predictor_of_interest'] == pred) &
                    (all_results['model']                 == 'Model4') &
                    (all_results['hemisphere']            == 'combined') &
                    (all_results['band']                  == band) &
                    (all_results['col']                   == col_key)
                ]
                if row.empty:
                    betas.append(np.nan); errors.append(0); pvals.append(1.0)
                else:
                    r = row.iloc[0]
                    betas.append(r['beta'])
                    errors.append(r['se'])
                    pvals.append(r.get('p_fdr', r['p_raw']))

            plot_betas  = [b if np.isfinite(b) else 0 for b in betas]
            plot_errors = [e if np.isfinite(betas[i]) else 0
                           for i, e in enumerate(errors)]

            bars = ax1.bar(x + offset, plot_betas, width,
                           color=color,
                           yerr=plot_errors,
                           error_kw=dict(ecolor=TEXT_COLOR, capsize=3,
                                          elinewidth=1),
                           alpha=0.78,
                           label=term_label)

            for bar, beta, p in zip(bars, betas, pvals):
                if not np.isfinite(beta):
                    continue
                stars = sig_stars(p)
                if stars:
                    h    = bar.get_height()
                    sign = 1 if h >= 0 else -1
                    ax1.text(bar.get_x() + bar.get_width() / 2,
                             h + sign * max(abs(h) * 0.05, 0.005),
                             stars, ha='center', va='bottom',
                             color=TEXT_COLOR, fontsize=9, fontweight='bold')

        ax1.axhline(0, color=SPINE_COLOR, lw=1.0, ls='--')
        ax1.set_xticks(x)
        ax1.set_xticklabels([b.capitalize() for b in BANDS], fontsize=9)
        ax1.set_xlabel('Frequency Band', fontsize=9)
        ax1.set_ylabel('β', fontsize=9)
        ax1.set_title(
            f"RSA_r ~ {PRED_LABELS[pred]}\n"
            f"[Model 4 — main effect (LHP) + hemisphere interaction]",
            fontsize=9, fontweight='bold')
        ax1.legend(facecolor='white', edgecolor=SPINE_COLOR, fontsize=8)

    fig.suptitle(
        'RSA_r ~ T_lag / SP_lag  |  β by Frequency Band  (HC only)',
        fontsize=13, fontweight='bold', y=1.01, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"  ✓ Beta bar chart → {save_path.name}")


# ============================================================================
# DATA LOADING
# ============================================================================

def load_data() -> Optional[pd.DataFrame]:
    dfs = []
    for exp in EXPERIMENTS:
        # Primary: combined HC all-bands file from rsa_lag_hc_bands.py
        fpath = INPUT_DIR / f"ALL_SUBJECTS_{exp}_hc_allbands_rsa_lag.csv"
        if fpath.exists():
            tmp = pd.read_csv(fpath)
            tmp['experiment'] = exp
            dfs.append(tmp)
            print(f"  Loaded {fpath.name}  ({len(tmp):,} rows)")
        else:
            # Fallback: stitch together per-region × per-band files
            region_band_dfs = []
            for region in ['LHP', 'RHP']:
                for band in BANDS:
                    rfpath = (INPUT_DIR /
                              f"ALL_SUBJECTS_{exp}_{region}_{band}_rsa_lag.csv")
                    if rfpath.exists():
                        tmp = pd.read_csv(rfpath)
                        tmp['experiment'] = exp
                        region_band_dfs.append(tmp)
            if region_band_dfs:
                combined = pd.concat(region_band_dfs, ignore_index=True)
                dfs.append(combined)
                print(f"  Reconstructed {exp} from per-region/band files "
                      f"({len(combined):,} rows)")
            else:
                print(f"  ✗ No data found for {exp} — skipping")

    if not dfs:
        return None

    df = pd.concat(dfs, ignore_index=True)

    # Ensure hemisphere column (Script 1 writes it, but guard for older files)
    if 'hemisphere' not in df.columns:
        HEMI_MAP = {'LHP': 'left', 'RHP': 'right'}
        df['hemisphere'] = df['region'].map(HEMI_MAP)

    # Prefix subjects to avoid cross-experiment ID collisions
    df['subject'] = df['experiment'].astype(str) + '_' + df['subject'].astype(str)

    # hemisphere_dummy: LHP=0 (reference), RHP=1
    df['hemisphere_dummy'] = (df['hemisphere'] == 'right').astype(int)

    # experiment_dummy: DBOY1=0 (reference)
    ref_exp = ('DBOY1' if 'DBOY1' in df['experiment'].unique()
               else df['experiment'].unique()[0])
    df['experiment_dummy'] = (df['experiment'] != ref_exp).astype(int)

    print(f"\n  Total rows   : {len(df):,}")
    print(f"  Subjects     : {df['subject'].nunique()}")
    print(f"  Bands present: {sorted(df['band'].unique().tolist())}")
    print(f"  Regions      : {sorted(df['region'].unique().tolist())}")

    return df


# ============================================================================
# ANALYSIS LOOP  (one predictor × one band)
# ============================================================================

def run_analysis_for_band(df: pd.DataFrame,
                           predictor: str,
                           band: str,
                           has_semsim: bool) -> Tuple[pd.DataFrame, str]:
    """
    Fits Models 1–5 for RSA_r ~ predictor within one frequency band,
    then fits Models 1 & 2 separately per hemisphere (LHP / RHP).
    """
    sub = df[df['band'] == band].copy()
    if sub.empty:
        return pd.DataFrame(), ""

    # ── Z-score outcome + continuous predictors within each subject ──────────
    # Columns to z-score: RSA_r, both lag predictors, output_lag covariate.
    # _z suffix columns are used in all LMM formulas; originals kept for plots.
    Z_COLS = ['RSA_r', 'T_lag', 'SP_lag', 'output_lag']
    sub = zscore_within_subject(sub, Z_COLS)

    # Convenience: map predictor name → its z-scored column name
    pred_z       = f'{predictor}_z'
    cross_cov    = CROSS_COVARIATE[predictor]
    cross_cov_z  = f'{cross_cov}_z'
    cross_label  = PRED_LABELS[cross_cov]
    output_lag_z = 'output_lag_z'

    # Override OUTCOME for formula building to use z-scored column
    outcome_col = 'RSA_r_z'

    print(f"\n  ── {outcome_col} ~ {pred_z}  |  {band.capitalize()} band  "
          f"──────────────────────────")
    print(f"     Rows: {len(sub):,}  |  Subjects: {sub['subject'].nunique()}")
    print(f"     Z-scored within subject: {Z_COLS}")

    all_rows    = []
    text_blocks = [
        f"OUTCOME: {outcome_col} (z-scored within subject)   "
        f"PREDICTOR: {pred_z}   BAND: {band}",
        f"N rows: {len(sub):,}  |  subjects: {sub['subject'].nunique()}\n",
    ]

    def run_and_collect(real_cols, model_key, title,
                        pred_display, hemisphere='combined',
                        data=None, formula_rhs=None):
        d = data if data is not None else sub

        # Temporarily rename outcome column so fit_lmm's hardcoded OUTCOME
        # ('RSA_r') resolves to the z-scored version
        d = d.copy()
        d[OUTCOME] = d[outcome_col]   # overwrite RSA_r with RSA_r_z in this copy

        res, _ = fit_lmm(d, real_cols, model_key, formula_rhs=formula_rhs)
        rows   = extract_rows(res, pred_display, model_key,
                              predictor, band, hemisphere)
        rows   = apply_fdr_within_model(rows)
        if not rows.empty:
            all_rows.append(rows)
            block = format_block(f"{title}  [{hemisphere}]", rows)
            text_blocks.append(block)
            print('\n' + block)
        return res

    int_term = f'{pred_z}:hemisphere_dummy'

    # ── Model 1: bare ────────────────────────────────────────────────────────
    run_and_collect(
        [pred_z, 'experiment_dummy'],
        'Model1',
        f'Model 1 — {outcome_col} ~ {pred_z} + experiment',
        {pred_z:             f'{PRED_LABELS[predictor]}  [z]',
         'experiment_dummy': 'experiment'},
    )

    # ── Model 2: controlled (cross-lag covariate also z-scored) ──────────────
    ctrl_cols = [pred_z, cross_cov_z, output_lag_z, 'experiment_dummy']
    ctrl_disp = {pred_z:             f'{PRED_LABELS[predictor]}  [z]',
                 cross_cov_z:        f'{cross_label}  [z, covariate]',
                 output_lag_z:       'output_lag  [z]',
                 'experiment_dummy': 'experiment'}
    if has_semsim:
        ctrl_cols = [pred_z, cross_cov_z, output_lag_z,
                     'semantic_sim', 'experiment_dummy']
        ctrl_disp['semantic_sim'] = 'semantic_sim'
    run_and_collect(ctrl_cols, 'Model2',
                    f'Model 2 — {outcome_col} ~ {pred_z} + {cross_cov_z} + controls',
                    ctrl_disp)

    # ── Model 3: hemisphere main effect ──────────────────────────────────────
    run_and_collect(
        [pred_z, 'hemisphere_dummy', 'experiment_dummy'],
        'Model3',
        f'Model 3 — {outcome_col} ~ {pred_z} + hemisphere',
        {pred_z:             f'{PRED_LABELS[predictor]}  [z]',
         'hemisphere_dummy': 'hemisphere (RHP vs LHP)',
         'experiment_dummy': 'experiment'},
    )

    # ── Model 4: predictor × hemisphere interaction ──────────────────────────
    run_and_collect(
        [pred_z, 'hemisphere_dummy', 'experiment_dummy'],
        'Model4',
        f'Model 4 — {outcome_col} ~ {pred_z} * hemisphere',
        {pred_z:             f'{PRED_LABELS[predictor]}  [z, LHP]',
         'hemisphere_dummy': 'hemisphere (RHP vs LHP)',
         int_term:           f'{pred_z} × hemisphere',
         'experiment_dummy': 'experiment'},
        formula_rhs=f'{pred_z} * hemisphere_dummy + experiment_dummy',
    )

    # ── Model 5: full + hemisphere interaction + cross-lag covariate ─────────
    base_real = [pred_z, cross_cov_z, 'hemisphere_dummy',
                 output_lag_z, 'experiment_dummy']
    base_rhs  = (f'{pred_z} * hemisphere_dummy '
                 f'+ {cross_cov_z} + {output_lag_z} + experiment_dummy')
    base_disp = {
        pred_z:             f'{PRED_LABELS[predictor]}  [z, LHP]',
        cross_cov_z:        f'{cross_label}  [z, covariate]',
        'hemisphere_dummy': 'hemisphere (RHP vs LHP)',
        int_term:           f'{pred_z} × hemisphere',
        output_lag_z:       'output_lag  [z]',
        'experiment_dummy': 'experiment',
    }
    if has_semsim:
        base_real = [pred_z, cross_cov_z, 'hemisphere_dummy',
                     output_lag_z, 'semantic_sim', 'experiment_dummy']
        base_rhs  = (f'{pred_z} * hemisphere_dummy '
                     f'+ {cross_cov_z} + {output_lag_z} + semantic_sim + experiment_dummy')
        base_disp['semantic_sim'] = 'semantic_sim'
    run_and_collect(
        base_real, 'Model5',
        f'Model 5 — {outcome_col} ~ {pred_z} * hemisphere + {cross_cov_z} + controls',
        base_disp,
        formula_rhs=base_rhs,
    )

    # ── Models 1 & 2 separately per hemisphere (LHP / RHP) ───────────────────
    for hemi in ['left', 'right']:
        hd = sub[sub['hemisphere'] == hemi].copy()
        if len(hd) < 20:
            continue

        run_and_collect(
            [pred_z, 'experiment_dummy'],
            'Model1',
            f'Model 1 — {outcome_col} ~ {pred_z} + experiment',
            {pred_z:             f'{PRED_LABELS[predictor]}  [z]',
             'experiment_dummy': 'experiment'},
            hemisphere=hemi, data=hd,
        )

        h_ctrl_cols = [pred_z, cross_cov_z, output_lag_z, 'experiment_dummy']
        h_ctrl_disp = {pred_z:             f'{PRED_LABELS[predictor]}  [z]',
                       cross_cov_z:        f'{cross_label}  [z, covariate]',
                       output_lag_z:       'output_lag  [z]',
                       'experiment_dummy': 'experiment'}
        if has_semsim:
            h_ctrl_cols = [pred_z, cross_cov_z, output_lag_z,
                           'semantic_sim', 'experiment_dummy']
            h_ctrl_disp['semantic_sim'] = 'semantic_sim'
        run_and_collect(
            h_ctrl_cols, 'Model2',
            f'Model 2 — {outcome_col} ~ {pred_z} + {cross_cov_z} + controls',
            h_ctrl_disp,
            hemisphere=hemi, data=hd,
        )

    result_df = (pd.concat(all_rows, ignore_index=True)
                 if all_rows else pd.DataFrame())
    return result_df, '\n\n'.join(text_blocks)


# ============================================================================
# ENTRY POINT
# ============================================================================

def main():
    print(f"\n{'='*80}")
    print("RSA_r ~ T_lag / SP_lag  |  HC only — by Frequency Band")
    print(f"{'='*80}")

    df = load_data()
    if df is None or df.empty:
        print("No data loaded. Exiting.")
        return

    has_semsim = ('semantic_sim' in df.columns and
                  df['semantic_sim'].notna().any())
    print(f"\n  Outcome    : RSA_r_z  (z-scored within subject, within band)")
    print(f"  Predictors : T_lag_z, SP_lag_z  (z-scored within subject, within band)")
    print(f"  Bands      : {BANDS}")
    print(f"  semantic_sim available: {has_semsim}")

    ALL_RESULTS = []

    for predictor in PREDICTORS:
        for band in BANDS:
            res_df, text = run_analysis_for_band(
                df, predictor, band, has_semsim)

            if not res_df.empty:
                ALL_RESULTS.append(res_df)

            tag      = f"{predictor}_{band}"
            csv_path = OUTPUT_DIR / f"LMM_{tag}_results.csv"
            txt_path = OUTPUT_DIR / f"LMM_{tag}_results.txt"
            if not res_df.empty:
                res_df.to_csv(csv_path, index=False)
                print(f"    ✓ {csv_path.name}")
            if text:
                with open(txt_path, 'w') as f:
                    f.write(text)
                print(f"    ✓ {txt_path.name}")

    if not ALL_RESULTS:
        print("  No results generated.")
        return

    master_df = pd.concat(ALL_RESULTS, ignore_index=True)
    master_df.to_csv(OUTPUT_DIR / 'LMM_ALL_results.csv', index=False)
    print(f"\n  ✓ Master results → LMM_ALL_results.csv  ({len(master_df):,} rows)")

    # ── Plots ─────────────────────────────────────────────────────────────────
    print("\n  Generating plots …")

    for predictor in PREDICTORS:
        for band in BANDS:
            plot_forest(
                master_df, predictor, band,
                PLOT_DIR / f"forest_{predictor}_{band}.png")
            plot_interaction(
                master_df, predictor, band, df,
                PLOT_DIR / f"interaction_{predictor}_{band}.png")

    plot_summary_heatmap(master_df, PLOT_DIR / 'summary_heatmap.png')
    plot_beta_bars(master_df,       PLOT_DIR / 'beta_bars_all.png')

    print(f"\n{'='*80}")
    print("✓ ANALYSIS COMPLETE")
    print(f"  Results : {OUTPUT_DIR}")
    print(f"  Plots   : {PLOT_DIR}")
    print(f"{'='*80}")


if __name__ == '__main__':
    main()


RSA_r ~ T_lag / SP_lag  |  HC only — by Frequency Band
  Loaded ALL_SUBJECTS_DBOY1_hc_allbands_rsa_lag.csv  (13,028 rows)
  Loaded ALL_SUBJECTS_EFRCourierReadOnly_hc_allbands_rsa_lag.csv  (1,712 rows)
  Loaded ALL_SUBJECTS_EFRCourierOpenLoop_hc_allbands_rsa_lag.csv  (1,276 rows)

  Total rows   : 16,016
  Subjects     : 44
  Bands present: ['alpha', 'beta', 'gamma', 'theta']
  Regions      : ['LHP', 'RHP']

  Outcome    : RSA_r_z  (z-scored within subject, within band)
  Predictors : T_lag_z, SP_lag_z  (z-scored within subject, within band)
  Bands      : ['theta', 'alpha', 'beta', 'gamma']
  semantic_sim available: True

  ── RSA_r_z ~ T_lag_z  |  Theta band  ──────────────────────────
     Rows: 4,004  |  Subjects: 44
     Z-scored within subject: ['RSA_r', 'T_lag', 'SP_lag', 'output_lag']
    [Model1] RSA_r ~ T_lag_z + experiment_dummy  |  N=4,004
    [Model1] optimizer=lbfgs  converged=True  llf=-5867.4724  AIC=nan

Model 1 — RSA_r_z ~ T_lag_z + experiment  [combined]
------------